# Ranker cross-encoder

This Kaggle notebook creates a trained cross-encoder using hyper-parameter optimization (HPO) followed by training using the best hyper-parameters found.  Metrics are calculated on the trained model with the test dataset.

The Ranker is a cross-encoder tier for a recommendation system designed for use with the [Movie Lens](https://files.grouplens.org/datasets/movielens/) 1M dataset and trained [TwoTowerDNN](https://www.kaggle.com/code/nicholeasuniquename/recommender-systems-with-tfx-pipelines/)

This notebooks' Ranker cross-encoder is designed using the [JAX AI stack](https://jaxstack.ai/).  It's a Flax NNX model that has a base layer instance of jraphx.nn.GATv2Conv
  which is the [jraphx](https://dirt.design/jraphx/) implementation of the
  Brody, Alon, Yahav ["How Attentive are Graph Attention Networks"](https://arxiv.org/pdf/2105.14491)
  model called "Universal Graph Attention".
  The GATv2 layer:
  * has dynamic attention that is dependent upon the query.
  * has a runtime complexity is O(V*F + E*F) where
    V is number of nodes, E is number of edges and F is number of
    features.
The last layer of the model is a score layer created from a Dense layer with output being equal to the number of candidate movie ids.
    
The graph Ranker model takes in a user's query graph which is the user query
    enriched by the user's watch history and candidate movies.
* During training, the candidate movies are built from negatives for the
    constrastive listwise learning.
* During inference, the candidate movies are given from the retrieval
    stage of the recommendation system.
* Padding is used to enable same shape inputs and outputs to jitted methods to use XLA without frequent re-compiles.
    
The model learns not only the relationships between 
users and movies, but also to rank them.

Part 1 of the [recommendation system](https://github.com/nking/recommender_systems.git) is
a Two-tower bi-encoder + Scann or Usearch for [retrieval](https://github.com/nking/retrieval.git),
part 2 is the [ranker](https://github.com/nking/ranker.git) (list-wise cross-encoder) in this project,
and part 3 is the [re-ranker](https://github.com/nking/reranker.git) (fine-tuned pretrained list-wise cross-encoder).
The end-to-end inference using the trained models from the bi-encoder and cross-encoder are in the ranker github project wrapped in rust and configured for gRPC (see [end-ro-end inference](https://github.com/nking/ranker/blob/development/inference_src/test/rust/test/app_config_test.rs).)

The Kaggle notebooks for the recommender systems of this system are at:
* [Recommender Systems](https://github.com/nking/reranker.git)
* [The TFX MLOps Pipeline](https://www.kaggle.com/code/nicholeasuniquename/recommender-systems-with-tfx-pipelines/)
* [Ranker](https://www.kaggle.com/code/nicholeasuniquename/ranker-cross-encoder-w-gatv2)
* [Re-Ranker](https://www.kaggle.com/code/nicholeasuniquename/re-ranker-fine-tuning-of-pre-trained)

Below is the ranker app HPO, train, test pipeline running on Kaggle T4x2 GPUs.  
* Jax uses XLA (accelerate linear algebra) to precompile computation graphs, fuse together operations like addition, multiplication, activation into a single kernel, and optimize graphs for machine instuctions so that off loading the calculations to GPUs uses less bandwidth and fewer crossings to/from the CPU to GPU interconnect.
* The use of Kaggle GPUs, specifically the T4x2 GPUs, use shared memory.  They each get a portion of the data to be processed and can write results to each other's VRAM.  The communication does not use PCIe-to-CPU or network cables.
* The training function in the ranker-app code uses jax to partition the data among the local devices (GPUs) and uses prefetching to place data onto GPUs ahead of time to keep the GPUs busy.  When the GPUs are used their load is between 50-100%.

Below bash scripting was chosen for running the dbs in the background. With some refactoring for python, running the app afterwards could be improved by rewriting the large bash shells for reusable python scripts...
Note that Gemini was used for essentially all of the bash code here with increasingly less back-and-forth.

The results from the pipelines are at the bottom of this page.

## CPU or GPUs

### Arithmetic intensity and ridge point for understanding when to use GPUs for training and inference.

* arithmetic intensity = flops / (data loads + stores in bytes)
* hardware ridge point = (peak hardware flops/sec) / (peak hardware bandwidth bytes/sec)
    is hardware specific and measured (see [roofline analysis](https://docs.nersc.gov/tools/performance/roofline/))
    * nvidia T4 GPU:  81 GFLOPS / 320 GBs
        **= 25 FLOPs/Byte** 
    * kaggle CPU: 100 GFLOPS / 20 GBs
        **= 5 FLOPs/Byte**

The arithmetic intensity of the model processing shows whether
the hardware memory or logic gates are the processing bottleneck
by comparing the arithmetic intensity to the hardware ridge point.

For a typical set of GraphRanker parameters we have:

The number of trainable parameters is:<br/>
    {'gatv2': 181680, 'rating_embed': 96, 'score_head': 97, 'total_trainable_params': 181873}<br/>
    jax_graph_comp_dict_single={'max_nodes': 192, 'max_edges': 192, 'max_graphs': 3}<br/>
    jax_graph_comp_dict_batch_256={'max_nodes': 38720, 'max_edges': 38464, 'max_graphs': 258}


* arithmetic intensity = flops / (data loads + stores in bytes)

> w/ 32-bit floats and a model size of P=182_000 trainable params,
  
> Memory:  182_000 f32s * 4 bytes/f32bit = 728_000 Bytes
  
> Flops: (primarily gatv2)<br/>
>> 1 multiply op + 1 add op = 2)<br/>
>> and runtime complexity is O(P * (V + E))) => 2 * P * (V + E))<br/>
        
>> single inference:<br/>
>>> = 2 * 182_000 * (192 + 192)<br/>
>>> = 140 M Flops<br/>
  
>> batch 256 inference:<br/>
>>> double the computations to include backward pass:<br/>
>>> = 2 * 2 * 182_000 * (38720 + 38464)<br/>
>>> = 56_200 M Flops<br/>

> single inference arithmetic intensity:<br/>
>> 140 M Flops / 728_000 Bytes
>> **= 192 Flops/Byte**
    
> batch arithmetic intensity:
>> 56_200 M Flops / (256*728_000) Bytes where 256 for optimizer and intermediate states
>> **= 139 Flops/Byte**


Both the single inference and batch training are greater than
the hardware ridge points of 5 and 25, so on those hardware
the processing would be **compute bound**.


To improve the performance:
* changing the batch size will not help because the numerator
      and denominator of the arithmetic intensity both scale roughly
      with batch size.
* raise the hardware's compute ceiling
    * nvidia GPU Tensor Cores can use float16 or bfloat16 to execute
        these half-precision floats expoentially faster than FP32.
    * FP32 nvidia hardware peak ~ 8.1 Terra Flops (10^12 floating point ops per sec)
    * FP16 nvidia hardware peak ~ 65 Terra Flops
        * cast the inputs and layer computations to FP16
             (the Jax ai stack is compat to half-precision floats, including export to savedmodel.  caveat is running half-precision on some CPUs)
* lower the total number of arithmetic operations
  * experiment w/ reducing GATv2 params: heads, hidden_features


#### single inference for GraphRanker:
* **choose CPU** because:
    * GPU communication cost:
        the latency in sending 728_000 Byte graph across PCIe bus
        and launching a CUDA kernel (~100µs to 1ms) plus GPU math time
        is about same or more than the time for CPU computation.
    * the computation on GPUs would not be anywhere near fully loading the GPUs
    * the cost of keeping a GPU serving always on is very high compared to CPU hosting.


#### batch training for GraphRanker:
* **choose GPU** because:
    * computation need is large, 56.2 GFlops
        * T4 GPU  hardware peak of 8.1 TFlops
        * CPU harward peak 100 GFlops
    * the GPUs ability to do math 81 times faster and in parallel
           efficiently even with the one time communication cost per batch
           results in smaller latency than the CPU even though the CPU has
           multicore processing and SIMD.

In general for a project one can use these guidelines:

* choose GPU if any of these are true:
    * arithmetic intensity > hardware ridge point, the processing is compute bound.
        if the processing # of FLOPs are large enough to make the CPU to GPU communication cost negigible in comparison, choose GPU (>= billions of FLOPS).
    * the dataset or model size is large enough to overburden the CPU RAM
        * choose GPU even if the process is memory bound
    * a deep learning frameworks with lots of linear algebra



In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

#import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%%bash
which nvidia-smi

In [ ]:
%%bash
ls -l
ls /tmp

## Install and run dbs in background threads

In [ ]:
%%bash
echo "install proot and curl"
apt-get update -y && apt-get install -y proot curl ca-certificates

date

export TMP=/tmp

export POSTGRES_USER=runner
export POSTGRES_PASSWORD=password123
export UID_DB=1000
export GID_DB=1000
export GCSFS_ENDPOINT_URL=http://127.0.0.1:4443
export TENSORSTORE_GCS_HTTP_URL=http://127.0.0.1:4443
export STORAGE_EMULATOR_HOST=http://127.0.0.1:4443
export CLOUD_STORAGE_EMULATOR_HOST=http://127.0.0.1:4443
export CLOUD_STORAGE_EMULATOR_ENDPOINT=http://127.0.0.1:4443
export TENSORSTORE_GCS_NO_AUTH=1
export NO_GCE_CHECK=true
export GCS_LAMBDA_TOKEN=none
export GOOGLE_AUTH_SUPPRESS_CREDENTIALS_WARNINGS=true
export STORAGE_EMULATOR_HOST_HTTP=true
export GOOGLE_AUTH_EXTERNAL_ACCOUNT_TOKEN_PROHIBIT=true
export GOOGLE_APPLICATION_CREDENTIALS=
export TENSORSTORE_GCS_REQUEST_CONCURRENCY=32

if true; then
    # was using fsouza/fake-gcs-server:1.54.0. using file paths instead now
    if [ ! -d "$TMP/gcs_rootfs" ]; then
        echo "create fake gcs server directory structure and copy in the data"
        mkdir -p $TMP/gcs_rootfs
        mkdir -p $TMP/gcs_data
        mkdir -p $TMP/gcs_data/data
        mkdir -p $TMP/gcs_data/hpo-results-bucket
        mkdir -p $TMP/gcs_data/mlflow_artifact_bucket
        mkdir -p $TMP/gcs_data/checkpoint-bucket
        cp -r /kaggle/input/datasets/nicholeasuniquename/ml-ranker-dataset2/* $TMP/gcs_data/data/
        #unzip /kaggle/input/datasets/nicholeasuniquename/ml-ranker-dataset2/ml-ranker-dataset2.zip.bin -d $TMP/gcs_data/data/
        chmod -R 775 $TMP/gcs_data
        DATA_DIR=$TMP/gcs_data/data
        LOG_FILE="/kaggle/working/data_manifest.txt"
        # Generates: <size_in_bytes>  <md5_hash>  <filepath>
        find "$DATA_DIR" -type f ! -name "*.log" | while read -r f; do
            printf "%s  %s  %s\n" \
                "$(stat -c %s "$f")" \
                "$(md5sum "$f" | awk '{print $1}')" \
                "$f"
        done | tee "$LOG_FILE"
    fi
fi

if true; then
    #vizier db:
    if [ ! -d "$TMP/vizier_rootfs" ]; then
        echo "unpack vizier tar"
        mkdir -p $TMP/vizier_rootfs/app/data
        tar -xf /kaggle/input/datasets/nicholeasuniquename/vizier-server/vizier-server.tar \
            -C $TMP/vizier_rootfs
    fi
    # Launch the server in the background (&)
    # -r sets the root directory to our extracted tarball
    # -b binds the host's $TMP so the container and main app share it
    # -w sets the working directory to /app (where your script lives)
    echo "run vizier server in the background"
    proot -r $TMP/vizier_rootfs -b $TMP:$TMP -b /dev:/dev \
        -b /proc:/proc \
        -w /app python3 run_vizier_server.py \
        > $TMP/vizier.log 2>&1 &

    echo "check that proot succeeded for vizier server"
    PROOT_PID_VIZIER=$!
    echo $PROOT_PID_VIZIER > $TMP/vizier_process_pid.txt

    sleep 5
    # kill -0 checks if a process exists without actually killing it
    if kill -0 $PROOT_PID_VIZIER 2>/dev/null; then
        echo "✅ Success: proot container is running in the background (PID VIZIER: $PROOT_PID_VIZIER)"
    else
        echo "❌ Failure: proot exited immediately. Check your paths or python dependencies."
        exit 1
    fi
    echo "Testing network connectivity to Vizier..."
    # Try up to 3 times with a 1-second delay
    for i in {1..3}; do
        #use a substitue for gRPC call:
        if curl -v --http2-prior-knowledge http://127.0.0.1:8080 2>&1 | grep -i "offering" || true; then
            echo "✅ Success: Vizier server answered successfully on port 8080!"
            break
        else
            echo "Attempt $i: Server not responding yet..."
            sleep 5
        fi
    done
    ps -aef | grep $PROOT_PID_VIZIER
    lsof -i :8080
    # ====> Vizier uri will be 127.0.0.1:8080
fi

if true; then
    #postgres:12-alpine
    if [ ! -d "$TMP/postgres_rootfs" ]; then
        echo "unpack postgres tar"
        mkdir -p $TMP/postgres_rootfs
        tar -xf /kaggle/input/datasets/nicholeasuniquename/pg-db-tar/pg-db.tar -C $TMP/postgres_rootfs
    fi
    echo "run postgres server in background"
    mkdir -p $TMP/pg_data
    chown -R 70:70 $TMP/pg_data 2>/dev/null || echo "Note: Cannot chown, running as user."

    # Check if the database has already been initialized
    if [ ! -f "$TMP/pg_data/PG_VERSION" ]; then
        echo "Database uninitialized. Running initdb..."
        env PROOT_NO_SECCOMP=1 \
            POSTGRES_USER=runner \
            POSTGRES_PASSWORD=password123 \
            POSTGRES_DB=mlflow_db \
            PGDATA=/var/lib/postgresql/data \
        proot \
            -r $TMP/postgres_rootfs \
            -i 70:70 \
            -b $TMP/pg_data:/var/lib/postgresql/data \
            /usr/local/bin/initdb -D /var/lib/postgresql/data -U runner

        # Ensure ownership is correct after initialization
        chown -R 70:70 $TMP/pg_data 2>/dev/null || true
    fi

    echo "Launching Postgres for MLFLow..."
    # Launching the binary directly bypasses all shell/entrypoint/su issues
    env PROOT_NO_SECCOMP=1 \
        POSTGRES_USER=runner \
        POSTGRES_PASSWORD=password123 \
        POSTGRES_DB=mlflow_db \
        PGDATA=/var/lib/postgresql/data \
    proot \
        -r $TMP/postgres_rootfs \
        -i 70:70 \
        -w /var/lib/postgresql/data \
        -b /dev:/dev -b /proc:/proc \
        -b $TMP/pg_data:/var/lib/postgresql/data \
        -b $TMP:$TMP \
        /usr/local/bin/postgres -D /var/lib/postgresql/data -c unix_socket_directories=$TMP \
        > $TMP/pg.log 2>&1 &

    echo "check that proot succeeded for postgres server"
    PROOT_PID_POSTGRES=$!
    echo $PROOT_PID_POSTGRES > $TMP/postgres_process_pid.txt
    if kill -0 $PROOT_PID_POSTGRES 2>/dev/null; then
        echo "✅ proot container launched (PID POSTGRES: $PROOT_PID_POSTGRES)"
    else
        echo "❌ Failure: proot exited immediately."
        cat $TMP/pg.log
        exit 1
    fi

    echo "Waiting for Postgres to accept connections..."
    for i in {1..20}; do
        if proot -r $TMP/postgres_rootfs -i 70:70 \
            -b $TMP:$TMP -b /dev:/dev /usr/local/bin/pg_isready \
            -h $TMP >/dev/null 2>&1; then
            echo "✅ Server is ready."
            break
        fi
        sleep 1
    done

    echo "Creating mlflow_db database..."
    env PROOT_NO_SECCOMP=1 \
        POSTGRES_USER=runner \
        POSTGRES_PASSWORD=password123 \
        POSTGRES_DB=mlflow_db \
        PGDATA=/var/lib/postgresql/data \
    proot \
        -r $TMP/postgres_rootfs \
        -i 70:70 \
        -b $TMP:$TMP \
        /usr/local/bin/createdb -h $TMP -U runner mlflow_db

    echo "Waiting for PostgreSQL to become responsive with mlflow_db..."
    for i in {1..30}; do
        # We run pg_isready INSIDE proot, pointing to the Unix socket directory (-h $TMP)
        if proot -r $TMP/postgres_rootfs -b $TMP:$TMP pg_isready -U runner -d mlflow_db -h $TMP >/dev/null 2>&1; then
            echo "✅ PostgreSQL is fully booted and accepting connections (verified via pg_isready)!"
            break
        fi
        
        sleep 1
        if [ $i -eq 30 ]; then
            echo "❌ Timed out waiting for Postgres. Printing last 15 lines of system log:"
            tail -n 15 $TMP/pg.log
            exit 1
        fi
    done

    pip install SQLAlchemy==2.0.49 psycopg2-binary==2.9.12 -q

    # The Ultimate Test: Can MLflow actually connect?
    echo "Testing MLflow SQLAlchemy URI connectivity..."
    python3 -c "
import os
import sqlalchemy
from sqlalchemy import text
tmp_dir = os.environ.get('TMP', '/tmp')
uri = f'postgresql+psycopg2://runner:password123@/mlflow_db?host={tmp_dir}'
try:
    engine = sqlalchemy.create_engine(uri)
    with engine.connect() as conn:
        result = conn.execute(text('SELECT 1')).scalar()
    print(f'✅ SUCCESS: Query result: {result}')
    print(f'✅ SUCCESS: MLflow can reach the database at {uri}.')
except Exception as e:
    print('❌ FAILURE: Could not connect to Postgres using the MLflow URI.')
    print(e)
"
    ps -aef | grep $PROOT_PID_POSTGRES
    lsof -i :5432
    # ====> mlflow uri is postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP
fi

echo "Note that proot info: default working directory is now / is $TMP/*_rootfs"

sleep 5


## unpack the ranker-app and install its dependendencies and itself into pip as packages

In [ ]:
%%bash
date
export TMP=/tmp

export RUSTUP_HOME=/kaggle/working/.rustup
export CARGO_HOME=/kaggle/working/.cargo
export PATH=$CARGO_HOME/bin:$PATH

#NOTE: if after the cargo instal, pip cannot find the cargo libraries,
# the notebook might need to be restarted and then use the following:
# source $HOME/.cargo/env && pip install . and the subsequent installs

# ranker-app
if [ ! -d "$TMP/app_rootfs" ]; then
    echo "unpack ranker-app tar"
    mkdir -p $TMP/app_rootfs
    tar -xzf /kaggle/input/datasets/nicholeasuniquename/ranker-app3/ra.tar.bin -C $TMP/app_rootfs
    cd $TMP/app_rootfs
    cp git_commit_hash.txt /kaggle/working/

    #install rust
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    source $CARGO_HOME/env

    #--no-deps due to jaxaistack deps
    pip install --no-deps -q -r requirements-kaggle-gpu.txt
    pip install .
    pip install jax[cuda12]==0.8.3 orbax-checkpoint==0.11.39
    pip list
    echo "if kaggle platform increase python version to 3.13 or greater, this cell will need to use a conda venv with python 3.12 and use the requirements-gpu.txt from the source code at github"
fi
date

## GPU check that both GPUs are communicating: looking at NCCL communication

Things to notice:
* Kaggle host is 1 cpu, so is not finding InfiniBand (IB) or RoCE-enabled network interface,
   but instead finds PCIe and defaults to using TCP/IP sockets over interface 'lo'
      NET/IB : No device found.
      NET/Socket : Using [0]lo:127.0.0.1<0>
      Initialized NET plugin Socket
* Data transfer between the GPUs (SHM/direct/direct) is using Shared Memory (SHM)
    ing 00 : 0 -> 1 -> 0 and Ring 01 : 1 -> 0 -> 1
    Channel 00 : 0[0] -> 1[1] via SHM/direct/direct
* NCCL_P2P_DISABLE=0 tries to establish a direct P2P PCIe first and falls back to    SHM (shared memory) upon failure.  But if the Kaggle virtualized environment (IOMMU or ACS) doesn't downgrade the connection and instead thinks it has a P2P path for GPU-to-GPU memory, this can lead to a deadlock when a JAX rendezvous attempts to push gradients across that path and doesn't receive an ackowledgement - the C++ thread spins infinitely waiting for that acknowledgment.
* NCCL_P2P_DISABLE=1 sets NCCL to bypass the P2P hardware probe.  JAX then immediately establishes an SHM ring (0 -> 1 -> 0) and routes all gradient synchronization through the CPU's system RAM, so is slower than a direct NVLink (which is not available on Kaggle), but it is stable and bypasses the hypervisor lockup.
* NCCL_IB_DISABLE=1 bypasses the NCCL probes for RoCE (RDMA over Converged Ethernet) and InfiniBand devices on startup.  Disabling the probes for nonexistant high speed network interfaces avoids a potential initialization hang and forces NCCL to use TCP/IP socket on the loopback interface.
* Use ring PXN 0 GDR 1:
    In the final configuration state, GDR (GPUDirect RDMA) is flagged, but Networking
  by Socket limits data movement to local memory copies.
  * if GPUDirect RDMA or NVLink were used data would travel directly from GPU0 to GPU1


## Check that ranker-app-gpu configuration is correct for T4x2 GPUs


In [ ]:
%%bash
export TMP=/tmp

echo "Starting Safe Native Execution Test..."
echo "---------------------------------------------------"

# We consolidate environment variables into the env call
timeout -k 50 45 env \
    NCCL_DEBUG=INFO \
    NCCL_DEBUG_SUBSYS=INIT,GRAPH,NET,COLL \
    NCCL_P2P_DISABLE=1 \
    NCCL_IB_DISABLE=1 \
    NCCL_SOCKET_IFNAME=lo \
    XLA_PYTHON_CLIENT_PREALLOCATE=false \
    python3 -u -c "
import jax
import jax.numpy as jnp
import os

print('Testing JAX devices...')
try:
    print('Devices:', jax.devices())
    # Force NCCL initialization
    x = jnp.ones(2)
    result = jax.pmap(lambda v: jax.lax.psum(v, axis_name='i'), axis_name='i')(x)
    print('--- SUCCESS! Result:', result, '---')
except Exception as e:
    print('--- FAILED! Exception:', e, '---')
"

EXIT_CODE=$?

echo "---------------------------------------------------"
if [ $EXIT_CODE -eq 124 ]; then
    echo "⚠️ TIMEOUT REACHED! The process hung and was safely killed."
elif [ $EXIT_CODE -eq 0 ]; then
    echo "✅ Process finished successfully!"
else
    echo "❌ Process failed with exit code $EXIT_CODE"
fi

In [ ]:
%%writefile set_env.sh
export TMP=/tmp

export project_id=kaggle-tune-train-test
export study_name=kaggle-hpo

# Clean the environment so gRPC doesn't route localhost to the internet
unset KAGGLE_DATA_PROXY_TOKEN
unset COLAB_KERNEL_MANAGER_PROXY_PORT
unset KAGGLE_DATA_PROXY_PROJECT
unset COLAB_LANGUAGE_SERVER_PROXY
unset KAGGLE_GRPC_DATA_PROXY_URL
unset KAGGLE_DATA_PROXY_URL
unset http_proxy
unset https_proxy
export POSTGRES_USER=runner
export POSTGRES_PASSWORD=password123
export UID_DB=1000
export GID_DB=1000
export GCSFS_ENDPOINT_URL=http://127.0.0.1:4443
export TENSORSTORE_GCS_HTTP_URL=http://127.0.0.1:4443
export STORAGE_EMULATOR_HOST=http://127.0.0.1:4443
export CLOUD_STORAGE_EMULATOR_HOST=http://127.0.0.1:4443
export CLOUD_STORAGE_EMULATOR_ENDPOINT=http://127.0.0.1:4443
export TENSORSTORE_GCS_NO_AUTH=1
export NO_GCE_CHECK=true
export GCS_LAMBDA_TOKEN=none
export GOOGLE_AUTH_SUPPRESS_CREDENTIALS_WARNINGS=true
export STORAGE_EMULATOR_HOST_HTTP=true
export GOOGLE_AUTH_EXTERNAL_ACCOUNT_TOKEN_PROHIBIT=true
export GOOGLE_APPLICATION_CREDENTIALS=
export TENSORSTORE_GCS_REQUEST_CONCURRENCY=32

# Force local networking to stay local
export no_proxy="localhost,127.0.0.1"

#export RSZ="small/"
export RSZ=""

In [ ]:
%%bash

echo "run gpu and connections check on ranker-app"

source set_env.sh

export project_id=kaggle
export study_name=kaggle-connections

#PYTHONPATH="${SITE_PACKAGES}" \

if true; then
    echo "run GPU and connections check"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --connections_check=1 \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            > gpu_check_output.log 2>&1 &

    GPU_CHECK_PID=$!
    echo $GPU_CHECK_PID > $TMP/gpu_check_output.txt
    echo "GPU CHECK started with PID $GPU_CHECK_PID. PID recorded to $TMP/gpu_check_output.txt"

    echo "Waiting for GPU CHECK process to complete..."
    wait $(cat $TMP/gpu_check_output.txt)
    echo "GPU CHECK finished"
    cat gpu_check_output.log
fi


## Pre-check for I/O bottlenecks.
is grain using enough threads and workers?
watch gpu and vram use.  would like to see GPU usage > 50%.  will explore using a training dataset of 1000 examples as a quick check.   Though te possiblity of memory induced swap thrashing isn't easy to see with this small number of epochs.

In [ ]:
%%bash

echo "prepare to run pre-check on HPO trial"

source set_env.sh

export project_id=kaggle
export study_name=kaggle-precheck

#if use much bigger dataset, can compare
#  grain_worker_count=0 \
#  grain_read_options_num_threads=1 \
#  grain_read_buffer_size=1 \
#to: 
#  grain_worker_count=2 \
#  grain_read_options_num_threads=4 \
#  grain_read_buffer_size=10 \

# /data/small holds 1000 examples, so be sure to keep batch_size < 1000

if true; then
    date
    echo "run HPO precheck"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=tune \
            --trial_ids='[0]' \
            --num_epochs=1 \
            --batch_size=512 \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --movies_uri=$TMP/gcs_data/data/movies.array_record \
            --ratings_train_liked_uri=$TMP/gcs_data/data/small/ratings_train_liked.array_record \
            --ratings_train_3_uri=$TMP/gcs_data/data/small/ratings_train_3.array_record \
            --ratings_train_disliked_uri=$TMP/gcs_data/data/small/ratings_train_disliked.array_record \
            --ratings_val_liked_uri=$TMP/gcs_data/data/small/ratings_val_liked.array_record \
            --ratings_val_3_uri=$TMP/gcs_data/data/small/ratings_val_3.array_record \
            --ratings_val_disliked_uri=$TMP/gcs_data/data/small/ratings_val_disliked.array_record \
            --ratings_test_liked_uri=$TMP/gcs_data/data/small/ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/small/ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/small/ratings_test_disliked.array_record \
            --recommendations_uri=$TMP/gcs_data/data/recommended_movies.array_record \
            --recommendations_ts_uri=$TMP/gcs_data/data/recommended_movies_timestamps.array_record \
            --movie_embeddings_uri=$TMP/gcs_data/data/movie_emb-00000-of-00001.array_record \
            --user_embeddings_uri=$TMP/gcs_data/data/user_emb-00000-of-00001.array_record \
            --movie_tiers_uri=$TMP/gcs_data/data/movie_tiers-00000-of-00001.array_record \
            --seed=12345 \
            --debug=True \
            > pre_hpo_output.log 2>&1 &

    PRE_HPO_PID=$!
    echo $PRE_HPO_PID > $TMP/pre_hpo_process_pid.txt
    echo "PRE HPO started with PID $PRE_HPO_PID. PID recorded to $TMP/pre_hpo_process_pid.txt"

    echo "Waiting for PRE HPO process to complete..."
    wait $(cat $TMP/pre_hpo_process_pid.txt)
    echo "PRE HPO finished"
fi



## Pre-check that orbax checkpoint save and restore succeed

In [ ]:
%%bash

echo "prepare to run pre-check on train"

source set_env.sh

export project_id=kaggle
export study_name=kaggle-precheck-orbax

#changed read uris from gcs uris to  using file paths instead:
# $TMP/gcs_data/hpo-results-bucket

if true; then
    date
    echo "run train precheck"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=train-given \
            --num_epochs=1 \
            --batch_size=512 \
            --train_id=456 \
            --validate_checkpoint_restores=True \
            --latest_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/latest \
            --best_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best \
            --top_k=20 \
            --num_layers=2 \
            --num_heads=4 \
            --hidden_dim=128 \
            --max_history=70 \
            --num_candidates=70 \
            --learning_rate=0.001 \
            --weight_decay=0.001 \
            --out_dim=32 \
            --edge_embed_dim=16 \
            --dropout_rate=0.2 \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --movies_uri=$TMP/gcs_data/data/movies.array_record \
            --ratings_train_liked_uri=$TMP/gcs_data/data/small/ratings_train_liked.array_record \
            --ratings_train_3_uri=$TMP/gcs_data/data/small/ratings_train_3.array_record \
            --ratings_train_disliked_uri=$TMP/gcs_data/data/small/ratings_train_disliked.array_record \
            --ratings_val_liked_uri=$TMP/gcs_data/data/small/ratings_val_liked.array_record \
            --ratings_val_3_uri=$TMP/gcs_data/data/small/ratings_val_3.array_record \
            --ratings_val_disliked_uri=$TMP/gcs_data/data/small/ratings_val_disliked.array_record \
            --ratings_test_liked_uri=$TMP/gcs_data/data/small/ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/small/ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/small/ratings_test_disliked.array_record \
            --recommendations_uri=$TMP/gcs_data/data/recommended_movies.array_record \
            --recommendations_ts_uri=$TMP/gcs_data/data/recommended_movies_timestamps.array_record \
            --movie_embeddings_uri=$TMP/gcs_data/data/movie_emb-00000-of-00001.array_record \
            --user_embeddings_uri=$TMP/gcs_data/data/user_emb-00000-of-00001.array_record \
            --movie_tiers_uri=$TMP/gcs_data/data/movie_tiers-00000-of-00001.array_record \
            --seed=12345 \
            --debug=True \
            > pre_train_output.log 2>&1 &

    PRE_TRAIN_PID=$!
    echo $PRE_TRAIN_PID > $TMP/pre_train_process_pid.txt
    echo "PRE TRAIN started with PID $PRE_TRAIN_PID. PID recorded to $TMP/pre_train_process_pid.txt"

    echo "Waiting for PRE TRAIN process to complete..."
    wait $(cat $TMP/pre_train_process_pid.txt)
    echo "PRE TRAIN finished.  check that orbax save and restore succeeded."
fi

if true; then
    date
    echo "run export-train-results"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=export-train-results \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --train_id=456 \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --latest_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/latest \
            --best_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best \
            --output_metrics_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/train/metrics.json" \
            --seed=12345 \
            --debug=True \
            > pre_exp_train_output.log 2>&1 &

    PRE_EXP_TRAIN_PID=$!
    echo $PRE_EXP_TRAIN_PID > $TMP/pre_export_train_process_pid.txt
    echo "PRE EXPORT TRAIN RESULTS started with PID $PRE_EXP_TRAIN_PID. PID recorded to $TMP/pre_exp_train_process_pid.txt"

    echo "Waiting for PRE EXPORT TRAIN RESULTS process to complete..."
    wait $(cat $TMP/pre_export_train_process_pid.txt)
    echo "PRE EXPORT TRAIN RESULTS finished"

fi

if true; then
    date
    echo "run test-given"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=test-given \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --test_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best/$study_name/train_456 \
            --test_id=456 \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --ratings_test_liked_uri=$TMP/gcs_data/data/small/ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/small/ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/small/ratings_test_disliked.array_record \
            --batch_size=512 \
            --seed=12345 \
            > pre_test_output.log 2>&1 &

    PRE_TEST_PID=$!
    echo $PRE_TEST_PID > $TMP/pre_test_process_pid.txt
    echo "PRE TEST started with PID $PRE_TEST_PID. PID recorded to $TMP/pre_test_process_pid.txt"

    echo "Waiting for PRE TEST process to complete..."
    wait $(cat $TMP/pre_test_process_pid.txt)
    echo "PRE TEST finished"
fi

if true; then
    date
    echo "run export-test-results"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=export-test-results \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --latest_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/latest \
            --best_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best \
            --output_metrics_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/test/metrics.json" \
            --seed=12345 \
            --debug=True \
            > pre_exp_test_output.log 2>&1 &

    PRE_EXP_TEST_PID=$!
    echo $PRE_EXP_TEST_PID > $TMP/pre_export_test_process_pid.txt
    echo "PRE EXPORT TEST RESULTS started with PID $PRE_EXP_TEST_PID. PID recorded to $TMP/pre_exp_test_process_pid.txt"

    echo "Waiting for PRE EXPORT TEST RESULTS process to complete..."
    wait $(cat $TMP/pre_export_test_process_pid.txt)
    echo "PRE EXPORT TEST RESULTS finished"
    cp -rf $TMP/gcs_data/hpo-results-bucket /kaggle/working/

fi

In [ ]:
%%bash
cat pre_train_output.log

In [ ]:
%%bash
find hpo-results-bucket -type f 
find hpo-results-bucket -type f | xargs less
date

## Run pipeline (HPO, train, test, extract)


### run tune

In [ ]:
%%bash

echo "prepare to run HPO trials with ranker-app"

#NOTE: changes regarding grain_read_buffer_size
# increase to 100 if long gaps between START_BATCH_TIME and the next START_BATCH_TIME
#   and gpu utiilization is low
# decrease to 10 or 20 if have OOM errors

source set_env.sh

##DEBUGGING: to temporarily set to small dataset, edit RSZ in set_env, and here edit num epochs and trials and RSZ

date

echo "reading ratings files such as: ", $TMP/gcs_data/data/${RSZ}ratings_train_liked.array_record

#prefer to use more trials when possible

#disabling to run "train-given" which runs train given bet parameters found by a previous HPO run
if true; then
    echo "run HPO trials"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=tune \
            --trial_ids='[0,1,2]' \
            --num_epochs=15 \
            --batch_size=1024 \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --movies_uri=$TMP/gcs_data/data/movies.array_record \
            --ratings_train_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_liked.array_record \
            --ratings_train_3_uri=$TMP/gcs_data/data/${RSZ}ratings_train_3.array_record \
            --ratings_train_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_disliked.array_record \
            --ratings_val_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_liked.array_record \
            --ratings_val_3_uri=$TMP/gcs_data/data/${RSZ}ratings_val_3.array_record \
            --ratings_val_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_disliked.array_record \
            --ratings_test_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/${RSZ}ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_disliked.array_record \
            --recommendations_uri=$TMP/gcs_data/data/recommended_movies.array_record \
            --recommendations_ts_uri=$TMP/gcs_data/data/recommended_movies_timestamps.array_record \
            --movie_embeddings_uri=$TMP/gcs_data/data/movie_emb-00000-of-00001.array_record \
            --user_embeddings_uri=$TMP/gcs_data/data/user_emb-00000-of-00001.array_record \
            --movie_tiers_uri=$TMP/gcs_data/data/movie_tiers-00000-of-00001.array_record \
            --seed=1234567 \
            > hpo_output.log 2>&1 &

    HPO_PID=$!
    echo $HPO_PID > $TMP/hpo_process_pid.txt
    echo "HPO started with PID $HPO_PID. PID recorded to $TMP/hpo_process_pid.txt"

    echo "Waiting for HPO process to complete..."
    wait $(cat $TMP/hpo_process_pid.txt)

    echo "HPO finished"
fi
date

### run tune export

In [ ]:
%%bash

source set_env.sh
date

#disabling to run "train-given" which runs train given bet parameters found by a previous HPO run

if true; then
    echo "extract HPO best params and metrics"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=export-hpo-results \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint="127.0.0.1:8080" \
            --output_hyperparams_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/tune/hparams.json" \
            --output_metrics_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/tune/metrics.json" \
            --debug=True \
            --seed=12345 \
            > export_hpo_output.log 2>&1 &

    EXP_HPO_PID=$!
    echo $EXP_HPO_PID > $TMP/export_hpo_process_pid.txt
    echo "HPO started with PID $EXP_HPO_PID. PID recorded to $TMP/export_hpo_process_pid.txt"

    echo "Waiting for Export HPO process to complete..."
    wait $(cat $TMP/export_hpo_process_pid.txt)
    echo "Export HPO finished"
    cp -rf $TMP/gcs_data/hpo-results-bucket /kaggle/working/
fi
date

### run train best

In [ ]:
%%bash

source set_env.sh

date

#disabling to run "train-given" which runs train given bet parameters found by a previous HPO run
if true; then
    echo "run train from best HPO hparams"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        JAX_PLATFORM_NAME=gpu \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        PYTHONIOENCODING=UTF-8 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=train-best \
            --num_epochs=15 \
            --batch_size=1024 \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --movies_uri=$TMP/gcs_data/data/movies.array_record \
            --ratings_train_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_liked.array_record \
            --ratings_train_3_uri=$TMP/gcs_data/data/${RSZ}ratings_train_3.array_record \
            --ratings_train_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_disliked.array_record \
            --ratings_val_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_liked.array_record \
            --ratings_val_3_uri=$TMP/gcs_data/data/${RSZ}ratings_val_3.array_record \
            --ratings_val_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_disliked.array_record \
            --ratings_test_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/${RSZ}ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_disliked.array_record \
            --recommendations_uri=$TMP/gcs_data/data/recommended_movies.array_record \
            --recommendations_ts_uri=$TMP/gcs_data/data/recommended_movies_timestamps.array_record \
            --movie_embeddings_uri=$TMP/gcs_data/data/movie_emb-00000-of-00001.array_record \
            --user_embeddings_uri=$TMP/gcs_data/data/user_emb-00000-of-00001.array_record \
            --movie_tiers_uri=$TMP/gcs_data/data/movie_tiers-00000-of-00001.array_record \
            --latest_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/latest \
            --best_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best \
            --seed=12345 \
            > train_output.log 2>&1 &

    TRAIN_PID=$!
    echo $TRAIN_PID > $TMP/train_process_pid.txt
    echo "TRAIN started with PID $TRAIN_PID. PID recorded to $TMP/train_process_pid.txt"

    echo "Waiting for TRAIN process to complete..."
    wait $(cat $TMP/train_process_pid.txt)
    echo "TRAIN finished"
    mkdir -p /kaggle/working/checkpoint-bucket
    cp -rf $TMP/gcs_data/checkpoint-bucket/best /kaggle/working/checkpoint-bucket/
fi
date

In [ ]:
%%bash

source set_env.sh

date

export sfx="${study_name}/train_789"

if false; then
    echo "run train given hparams"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        JAX_PLATFORM_NAME=gpu \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        PYTHONIOENCODING=UTF-8 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=train-given \
            --train_id="789" \
            --dropout_rate=0.1 \
            --edge_embed_dim=8 \
            --embed_in_dim=32 \
            --embed_len=32 \
            --focal_loss_gamma=2 \
            --hidden_dim=64 \
            --hpo_tier=0 \
            --learning_rate=0.001 \
            --max_history=100 \
            --mlp_hidden_dim=1.0 \
            --num_candidates=100 \
            --num_layers=2 \
            --num_heads=2 \
            --num_movies=3883 \
            --num_users=6040 \
            --out_dim=32 \
            --temperature=0.25 \
            --tier_weights='[0.33,0.33,0.33]' \
            --top_k=20 \
            --use_focal_loss=0 \
            --use_ipw=0 \
            --weight_decay=0.001 \
            --num_epochs=15 \
            --batch_size=1024 \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --movies_uri=$TMP/gcs_data/data/movies.array_record \
            --ratings_train_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_liked.array_record \
            --ratings_train_3_uri=$TMP/gcs_data/data/${RSZ}ratings_train_3.array_record \
            --ratings_train_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_disliked.array_record \
            --ratings_val_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_liked.array_record \
            --ratings_val_3_uri=$TMP/gcs_data/data/${RSZ}ratings_val_3.array_record \
            --ratings_val_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_disliked.array_record \
            --ratings_test_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/${RSZ}ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_disliked.array_record \
            --recommendations_uri=$TMP/gcs_data/data/recommended_movies.array_record \
            --recommendations_ts_uri=$TMP/gcs_data/data/recommended_movies_timestamps.array_record \
            --movie_embeddings_uri=$TMP/gcs_data/data/movie_emb-00000-of-00001.array_record \
            --user_embeddings_uri=$TMP/gcs_data/data/user_emb-00000-of-00001.array_record \
            --movie_tiers_uri=$TMP/gcs_data/data/movie_tiers-00000-of-00001.array_record \
            --latest_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/latest/ \
            --best_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best/ \
            --output_metrics_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/train/metrics.json" \
            --debug=True \
            --seed=12345 \
            > train_output.log 2>&1 &

    TRAIN_PID=$!
    echo $TRAIN_PID > $TMP/train_process_pid.txt
    echo "TRAIN started with PID $TRAIN_PID. PID recorded to $TMP/train_process_pid.txt"

    echo "Waiting for TRAIN process to complete..."
    wait $(cat $TMP/train_process_pid.txt)
    echo "TRAIN finished"
    mkdir -p /kaggle/working/checkpoint-bucket
    cp -rf $TMP/gcs_data/checkpoint-bucket/best /kaggle/working/checkpoint-bucket/
fi
date

### run train export

In [ ]:
%%bash

source set_env.sh

date
if true; then
    echo "run export-train-results"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=export-train-results \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --latest_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/latest \
            --best_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best \
            --output_metrics_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/train/metrics.json" \
            --seed=12345 \
            --debug=True \
            > exp_train_output.log 2>&1 &

    EXP_TRAIN_PID=$!
    echo $EXP_TRAIN_PID > $TMP/export_train_process_pid.txt
    echo "EXPORT TRAIN RESULTS started with PID $EXP_TRAIN_PID. PID recorded to $TMP/exp_train_process_pid.txt"

    echo "Waiting for EXPORT TRAIN RESULTS process to complete..."
    wait $(cat $TMP/export_train_process_pid.txt)
    echo "EXPORT TRAIN RESULTS finished"
    cp -rf $TMP/gcs_data/hpo-results-bucket /kaggle/working/
fi
date

### run test best

In [ ]:
%%bash

source set_env.sh

date

#disabling to run "test-given" which runs train given bet parameters found by a previous HPO run
if true; then
    echo "run test"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=test-best \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --movies_uri=$TMP/gcs_data/data/movies.array_record \
            --ratings_train_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_liked.array_record \
            --ratings_train_3_uri=$TMP/gcs_data/data/${RSZ}ratings_train_3.array_record \
            --ratings_train_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_disliked.array_record \
            --ratings_val_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_liked.array_record \
            --ratings_val_3_uri=$TMP/gcs_data/data/${RSZ}ratings_val_3.array_record \
            --ratings_val_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_disliked.array_record \
            --ratings_test_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/${RSZ}ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_disliked.array_record \
            --recommendations_uri=$TMP/gcs_data/data/recommended_movies.array_record \
            --recommendations_ts_uri=$TMP/gcs_data/data/recommended_movies_timestamps.array_record \
            --movie_embeddings_uri=$TMP/gcs_data/data/movie_emb-00000-of-00001.array_record \
            --user_embeddings_uri=$TMP/gcs_data/data/user_emb-00000-of-00001.array_record \
            --movie_tiers_uri=$TMP/gcs_data/data/movie_tiers-00000-of-00001.array_record \
            --batch_size=1024 \
            --seed=12345 \
            > test_output.log 2>&1 &

    TEST_PID=$!
    echo $TEST_PID > $TMP/test_process_pid.txt
    echo "TEST started with PID $TEST_PID. PID recorded to $TMP/test_process_pid.txt"

    echo "Waiting for TEST process to complete..."
    wait $(cat $TMP/test_process_pid.txt)
    echo "TEST finished"
fi
date

In [ ]:
%%bash

source set_env.sh

date

export sfx="${study_name}/train_789"

if false; then
    echo "run test-given"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        grain_worker_count=2 \
        grain_read_options_num_threads=4 \
        grain_read_buffer_size=10 \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=test-given \
            --train_id=789 \
            --test_id=789 \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --movies_uri=$TMP/gcs_data/data/movies.array_record \
            --ratings_train_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_liked.array_record \
            --ratings_train_3_uri=$TMP/gcs_data/data/${RSZ}ratings_train_3.array_record \
            --ratings_train_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_train_disliked.array_record \
            --ratings_val_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_liked.array_record \
            --ratings_val_3_uri=$TMP/gcs_data/data/${RSZ}ratings_val_3.array_record \
            --ratings_val_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_val_disliked.array_record \
            --ratings_test_liked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_liked.array_record \
            --ratings_test_3_uri=$TMP/gcs_data/data/${RSZ}ratings_test_3.array_record \
            --ratings_test_disliked_uri=$TMP/gcs_data/data/${RSZ}ratings_test_disliked.array_record \
            --recommendations_uri=$TMP/gcs_data/data/recommended_movies.array_record \
            --recommendations_ts_uri=$TMP/gcs_data/data/recommended_movies_timestamps.array_record \
            --movie_embeddings_uri=$TMP/gcs_data/data/movie_emb-00000-of-00001.array_record \
            --user_embeddings_uri=$TMP/gcs_data/data/user_emb-00000-of-00001.array_record \
            --movie_tiers_uri=$TMP/gcs_data/data/movie_tiers-00000-of-00001.array_record \
            --output_metrics_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/test/metrics.json" \
            --debug=True \
            --test_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best/$sfx \
            --batch_size=1024 \
            --seed=12345 \
            > test_output.log 2>&1 &

    TEST_PID=$!
    echo $TEST_PID > $TMP/test_process_pid.txt
    echo "TEST started with PID $TEST_PID. PID recorded to $TMP/test_process_pid.txt"

    echo "Waiting for TEST process to complete..."
    wait $(cat $TMP/test_process_pid.txt)
    echo "TEST finished"
fi
date

### run export test

In [ ]:
%%bash

source set_env.sh

date

if true; then
    echo "run export-test-results"
    env NCCL_P2P_DISABLE=1 \
        NCCL_IB_DISABLE=1 \
        NCCL_SOCKET_IFNAME=lo \
        XLA_PYTHON_CLIENT_PREALLOCATE=false \
        XLA_PYTHON_CLIENT_ALLOCATOR=platform \
        JAX_PLATFORM_NAME=gpu \
        PYTHONIOENCODING=UTF-8 \
        PYTHONUNBUFFERED=1 \
        PATH=/usr/local/bin:/usr/bin:/bin \
    /usr/bin/python3 -u -m movie_lens_ranker.app_runner \
            --phase=export-test-results \
            --LOGNAME=runner \
            --USER=runner \
            --study_name=$study_name \
            --project_id=$project_id \
            --mlflow_experiment_name=$study_name \
            --mlflow_tracking_uri="postgresql+psycopg2://runner:password123@/mlflow_db?host=$TMP" \
            --vizier_endpoint=127.0.0.1:8080 \
            --latest_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/latest \
            --best_checkpoint_uri=$TMP/gcs_data/checkpoint-bucket/best \
            --output_metrics_uri="$TMP/gcs_data/hpo-results-bucket/${project_id}/${study_name}/test/metrics.json" \
            --seed=12345 \
            --debug=True \
            > exp_test_output.log 2>&1 &

    EXP_TEST_PID=$!
    echo $EXP_TEST_PID > $TMP/export_test_process_pid.txt
    echo "EXPORT TEST RESULTS started with PID $EXP_TEST_PID. PID recorded to $TMP/exp_test_process_pid.txt"

    echo "Waiting for EXPORT TEST RESULTS process to complete..."
    wait $(cat $TMP/export_test_process_pid.txt)
    echo "EXPORT TEST RESULTS finished"
    cp -rf $TMP/gcs_data/hpo-results-bucket /kaggle/working/

fi

date

In [ ]:
%%bash
export TMP=/tmp

echo "Pipeline execution finished."
date

# --- Post-Processing ---
echo "copy HPO params and metrics to /kaggle/working"
cp -rf $TMP/gcs_data/hpo-results-bucket /kaggle/working/

echo "All tasks finished successfully!"

# --- Shutdown Services ---
stop_service() {
    local pid_file=$1    
    if [ ! -f "$pid_file" ]; then
        echo "PID file $pid_file does not exist. Skipping."
        return
    fi
    local pid=$(cat "$pid_file")
    if kill -0 "$pid" 2>/dev/null; then
        echo "Stopping process $pid..."
        kill -15 "$pid"
        for i in {1..5}; do
            if ! kill -0 "$pid" 2>/dev/null; then
                echo "Stopped gracefully."
                rm -f "$pid_file"
                return
            fi
            sleep 1
        done
        kill -9 "$pid"
        rm -f "$pid_file"
    else
        echo "Process $pid already dead. Cleaning file."
        rm -f "$pid_file"
    fi
}
date

stop_service "$TMP/vizier_process_pid.txt"
stop_service "$TMP/postgres_process_pid.txt"
#stop_service "$TMP/gcs_process_pid.txt"
echo "All services handled."

echo "closing any remaining open ports"
#fuser -k 4443/tcp 2>/dev/null || true
fuser -k 5432/tcp 2>/dev/null || true
fuser -k 1234/tcp 2>/dev/null || true
fuser -k 8080/tcp 2>/dev/null || true

In [ ]:
%%bash
rm -rf /kagglw/working/.cargo
rm -rf /kaggle/working/.rustup

## Plot the train and validation curves

%%bash
date
pip install -q plotly==5.24.1 kaleido==5.24.1
date5.24.1

import fsspec
import json
import os
import plotly.graph_objects as go

try:
    project_id="kaggle-tune-train-test"
    study_name="kaggle-hpo"
    output_metrics_path = f"/kaggle/working/hpo-results-bucket/{project_id}/{study_name}/train/metrics.json"
    
    with fsspec.open(output_metrics_path, mode='r') as f: 
        content = f.read() 
        output_metrics_dict = json.loads(content)
    
    out_dir = "/kaggle/working/training_curves"
    
    def plot_metrics(metrics_dict: dict, out_dir: str):
        os.makedirs(out_dir, exist_ok=True)
        
        for key in ["loss", 'ndcg_20', 'recall_20', 'mrr_20']:
            # Extract Polars Series directly out of the dictionary 
            epochs = metrics_dict[f'train_{key}']['x']
            train_vals = metrics_dict[f'train_{key}']['y']
            val_vals = metrics_dict[f'val_{key}']['y']
            
            # Build the figure with zero memory overhead
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=epochs, y=train_vals, mode='lines', name='train'))
            fig.add_trace(go.Scatter(x=epochs, y=val_vals, mode='lines', name='val'))
            
            fig.update_layout(
                title={
                    'text': f"{key.upper()} over Epochs",
                    'font': {'size': 22}
                },
                xaxis={
                    'title': 'epoch',
                    'title_font': {'size': 18},
                    'tickfont': {'size': 14}
                },
                yaxis={
                    'title': key,
                    'title_font': {'size': 18},
                    'tickfont': {'size': 14}
                },
                legend={
                    'font': {'size': 16}
                },
                template="plotly_white",
                width=800,
                height=500
            )
            
            # Save to PNG
            out_path = os.path.join(out_dir, f"{key}.png")
            fig.write_image(out_path, scale=2) # scale=2 doubles resolution (~200 PPI)
            
            #render in notebook:
            fig.show(renderer="iframe")
    
    plot_metrics(output_metrics_dict, out_dir)
except Exception as ex:
    print(f'Error: {ex}')

In [ ]:
%%bash
find hpo-results-bucket
echo "============================"
find  /tmp/gcs_data/hpo-results-bucket/

## Train, val, test curves (from the last trained model in the repository):

<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/composite_ndcg_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/ndcg_head_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/ndcg_torso_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/ndcg_tail_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/ndcg_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/recall_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/recall_head_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/recall_torso_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/recall_tail_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/precision_1.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/precision_5.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/precision_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/mrr_20.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/logit_max.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/logit_mean.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/logit_min.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/logit_std.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/training_metrics_pngs/loss.png">
### Movie Embedding UMAP plots
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/post_training_analysis_q01_r01/embedding_plots/gatv2_emb_umap_users_user_tier.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/post_training_analysis_q01_r01/embedding_plots/gatv2_emb_umap_movies_movie_tier.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/post_training_analysis_q01_r01/embedding_plots/gatv2_emb_umap_catalog_anchored_joint_users_movies.png">
<img src="https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/post_training_analysis_q01_r01/embedding_plots/gatv2_emb_umap_joint_users_movies.png">



## Post-training Analysis (from latest trained model in repository):

In [ ]:
import requests
from rich import print_json

url = "https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/post_training_analysis_q01_r01/summary_conclusions.json"
response = requests.get(url)

# Prints syntax-highlighted, nicely indented JSON
print_json(response.text)

url2 = "https://raw.githubusercontent.com/nking/ranker/refs/heads/main/docs/mlops/model_analysis/post_training_analysis_q01_r01/summary_metrics.json"
response = requests.get(url2)

# Prints syntax-highlighted, nicely indented JSON
print_json(response.text)

## What's next

The re-ranker tier is in the Kaggle notebook [Re-Ranker](https://www.kaggle.com/code/nicholeasuniquename/re-ranker-fine-tuning-of-pre-trained)
